<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🏷️</div><div style="color:white;font-size:1.5rem;font-weight:700;">Part-of-Speech Tagger</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Tokenise, lemmatise, and tag texts in 65+ languages<br><a href="https://ladal.edu.au/tutorials/pos_tagging/pos_tagging.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to <b>MyTexts</b> (Step 2). Demo texts are loaded automatically if the folder is empty.</li><li>Set the correct <b>language model</b> in the configuration cell (Step 2).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download the tagged Excel file and ZIP from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> Upload your files to the appropriate folder, edit the configuration cell, then click <b>Kernel &#x2192; Restart &amp; Run All</b>. The notebook runs from top to bottom automatically.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Step 1 — Load shared helpers (do not edit)
source("../helpers/ladal_helpers.R")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts &amp; configure</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left (click the folder icon if hidden).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b>.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. If the folder is empty, demo data will be loaded automatically.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────
# Set the language model, then run this cell (Shift+Enter).
#
# Common models (29 languages bundled — see full list at
#   https://ladal.edu.au/tools.html):
#   "english-ewt"       English (recommended)
#   "english-gum"       English (alternative)
#   "german-gsd"        German
#   "french-gsd"        French
#   "spanish-ancora"    Spanish
#   "italian-isdt"      Italian
#   "dutch-alpino"      Dutch
#   "portuguese-bosque" Portuguese
#   "russian-gsd"       Russian
#   "chinese-gsd"       Chinese
#   "arabic-padt"       Arabic
#   "japanese-gsd"      Japanese

language_model <- "english-ewt"


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
suppressPackageStartupMessages(library(udpipe))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(zip))

seed_demo_texts("MyTexts")

texts <- tryCatch(load_texts("MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })

if (!is.null(texts)) {
  show_corpus_summary(texts)
  .info("Large files may take several minutes to tag.")
  .prog("Loading language model...", 15)
  model_dir <- file.path(Sys.getenv("HOME", "/home/jovyan"), "udpipe-models")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  existing <- list.files(model_dir,
    pattern = paste0("^", language_model, ".*\\.udpipe$"),
    full.names = TRUE, recursive = TRUE)
  if (length(existing) > 0) {
    model_path <- existing[1]
    .ok(paste("Using cached model:", basename(model_path)))
  } else {
    .info("Downloading language model (30–60 seconds)...")
    dl <- udpipe_download_model(language = language_model, model_dir = model_dir)
    if (isTRUE(dl$download_failed))
      stop("Model download failed. Check your internet connection.")
    model_path <- dl$file_model
    .ok(paste("Downloaded:", basename(model_path)))
  }
  model <- udpipe_load_model(model_path)
  tagged_list <- lapply(seq_along(texts), function(i) {
    nm <- names(texts)[i]
    .prog(paste0("Tagging: ", nm, " (", i, "/", length(texts), ")"),
          20 + 65 * (i / length(texts)))
    as.data.frame(udpipe_annotate(model, x = texts[[nm]], doc_id = nm))
  })
  tagged_df <- dplyr::bind_rows(tagged_list) |>
    dplyr::select(doc_id, sentence_id, token_id, token, lemma,
                  upos, xpos, dep_rel, head_token_id) |>
    dplyr::rename(Document=doc_id, Sentence=sentence_id,
                  TokenID=token_id, Token=token, Lemma=lemma,
                  UPOS=upos, XPOS=xpos, DepRel=dep_rel, HeadTokenID=head_token_id)
  # Preview first 20 rows
  preview <- head(tagged_df[, c("Document","Token","Lemma","UPOS","XPOS")], 20)
  html_rows <- apply(preview, 1, function(r) paste0(
    "<tr>",
    paste(sapply(r, function(v) paste0(
      '<td style="padding:3px 10px;font-size:.82rem;font-family:monospace;">',
      htmltools::htmlEscape(as.character(v)), "</td>")), collapse=""),
    "</tr>"))
  suppressPackageStartupMessages(library(htmltools))
  headers <- c("Document","Token","Lemma","UPOS","XPOS")
  th <- paste(sapply(headers, function(h) paste0(
    '<th style="padding:3px 10px;text-align:left;font-size:.8rem;',
    'color:#51247a;border-bottom:2px solid #d0c4e8;">', h, "</th>")),
    collapse="")
  IRdisplay::display_html(paste0(
    '<div style="font-family:sans-serif;">',
    '<b style="color:#51247a;font-size:.88rem;">Preview (first 20 tokens)</b>',
    '<div style="overflow-x:auto;max-height:360px;overflow-y:auto;margin-top:6px;">',
    '<table style="border-collapse:collapse;width:100%;">',
    '<thead style="position:sticky;top:0;background:#f4f0f8;">',
    "<tr>", th, "</tr></thead><tbody>",
    paste(html_rows, collapse="\n"),
    "</tbody></table></div></div>"
  ))
  # POS frequency chart
  .prog("Plotting POS summary...", 88)
  pos_counts <- tagged_df |>
    dplyr::filter(!is.na(UPOS), !UPOS %in% c("PUNCT", "SPACE")) |>
    dplyr::count(UPOS, sort = TRUE)
  p <- ggplot(pos_counts, aes(x = reorder(UPOS, n), y = n)) +
    geom_col(fill = "#51247a", width = .7) +
    geom_text(aes(label = format(n, big.mark=",")),
              hjust = -0.1, size = 3.5, colour = "gray30") +
    coord_flip() +
    scale_y_continuous(expand = expansion(mult = c(0, .15))) +
    theme_bw(base_size = 12) +
    labs(title = "Token count by POS category",
         x = "Universal POS tag", y = "Count")
  print(p)
  save_excel(tagged_df, "pos_tagged.xlsx")
  dir.create("MyOutput", showWarnings=FALSE, recursive=TRUE)
  ggsave("MyOutput/pos_summary.png", bg="white", width=7, height=5, dpi=200)
  tmp <- tempfile(); dir.create(tmp)
  for (doc in unique(tagged_df$Document)) {
    df <- dplyr::filter(tagged_df, Document==doc, !is.na(Token), !is.na(UPOS))
    writeLines(paste(paste0(df$Token,"_",df$UPOS), collapse=" "),
               file.path(tmp, paste0(doc, "_postag.txt")))
  }
  zip::zip("MyOutput/pos_tagged_texts.zip",
           files=list.files(tmp, full.names=TRUE), mode="cherry-pick")
  .ok(paste0("Tagged <b>", nrow(tagged_df), " tokens</b>. ",
             "Saved: pos_tagged.xlsx, pos_summary.png, pos_tagged_texts.zip."))
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Files saved: <b>pos_tagged.xlsx, pos_summary.png, pos_tagged_texts.zip</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


## Citation

<div style="background:#faf7fd; border:1px solid #e0d4f0; border-left:5px solid #51247A; border-radius:6px; padding:16px 20px; margin:1.5rem 0; font-size:0.9rem; line-height:1.7; color:#333;">
  <div style="font-weight:700; color:#51247A; margin-bottom:8px;">&#x1F4CB; How to cite this tool</div>

Schweinberger, Martin. (2026). *LADAL Part-of-Speech Tagger* (Version 2.0.1). Brisbane: The University of Queensland. [https://ladal.edu.au/tools.html](https://ladal.edu.au/tools.html)

<details style="margin-top:12px;">
<summary style="cursor:pointer; font-size:0.82rem; color:#51247A; font-weight:600;">BibTeX</summary>

```bibtex
@software{schweinberger2026postagger,
  author       = {Schweinberger, Martin},
  title        = {LADAL Part-of-Speech Tagger},
  year         = {2026},
  version      = {2.0.1},
  organization = {The University of Queensland},
  url          = {https://ladal.edu.au/tools.html},
  doi          = {https://ladal.edu.au/tools.html}
}
```

</details>
</div>

In [ ]:
# Uncomment to show package versions for reproducibility
# sessionInfo()
